# 055 — Set up the cyclic pushover and modal analyses, one per design group

The regression covariates of nb `072` §8 need two structural properties that no existing analysis provides for
the 51 site-specific designs:

* **overstrength** $\Omega = V_{max}/V_{design}$, and
* **period-based ductility** $\mu_T = \delta_u/\delta_{y,eff}$,

both read off the **envelope (backbone) of a cyclic pushover curve**. A cyclic rather than a monotonic pushover is
used because the braces' capacity degrades under load reversals (buckling, then fracture): the envelope of a cyclic
test is the more realistic backbone for collapse under earthquake loading.

The existing cyclic pushovers of these designs (nb `011`) are unsuitable: they were set up per *site*, sit on the
analysis drive, some did not finish, and their fixed 250 mm amplitude (≈ 2.4 % roof drift for 3s, 1.4 % for 5s) is
too small to reach the post-peak 0.8 $V_{max}$ point. This notebook therefore **writes** a new FEMA 461 cyclic
pushover, with an amplitude set per structure from a roof-drift target, and a modal analysis for every design group.
**It runs nothing.** The analyses are run outside this project, and nb `056` post-processes them into the file nb
`072` reads.

## What this notebook writes

1. Into each group's analysis folder `DEST_ROOT/group_{n}s_{gid:02d}/{n}s/mdof/` (the same folders nb `052`/`054`
   use; `DEST_ROOT = analysis_data.wp1_fixed_record_sets`):
   * the design file and the full-recorder structural model (`initialise_model.py`) - byte-identical to what nb
     `052` writes, so an existing folder is simply refreshed;
   * **modal**: `run_modal.py` + `config_modal.py` (as nb `052`);
   * **cyclic pushover**: `run_cyclic_pushover.py` + `config_cyclic_pushover.py` (as nb `010` §4).
   Only where the analysis drive is attached: `BUILD_ANALYSIS_FOLDERS`.
2. Batch launchers `cpo.py` and `modal.py` per storey count, next to the existing IDA/MSA launchers in
   `phd_project/scripts/WP1_ground_motion_set/batch_run_analyses/fixed_record_sets/{n}s/mdof/`. They always carry
   the `DEST_ROOT` paths, whether or not the folders were built here.

## The cyclic pushover

* **Loading protocol**: FEMA 461 (quasi-static cyclic, `loading_protocols.FEMA_461_loading_protocol`). There are
  `CPO_N_STEPS` = 12 amplitude levels, each 1.4 × the previous, with two full cycles (±, ±) per level, ending at
  **± U_max** and then returning to 0.
* **Amplitude**: $U_{max}$ = `CPO_MAX_DRIFT` × roof height, set per structure (6 % roof drift → 630 mm for 3s,
  1050 mm for 5s), so every design is driven far enough past its peak to reach $0.8\,V_{max}$ on the envelope.
  Setting `CPO_MAX_DRIFT = None` falls back to `get_FEMA461_displacements_for_building`'s own estimate
  (1.2 × a regression-based residual displacement).
* **Control**: displacement-controlled at the roof control node (`get_control_node_from_design_file`: top-left roof
  node, 101010400 for 3s, 101010600 for 5s), after gravity (`loadConst`). The ramp is built from steps of `dU`
  (0.2 mm, as nb `010`). A finer `dU` already tuned on disk for a folder is **kept** (`cpo_du.resolve_cpo_du`),
  because some models only converge with one.
* **Load pattern**: **EC8 inverted-triangular** lateral load pattern, $F_i \propto m_i h_i$, normalised to sum to 1,
  so the load factor **is the base shear in N**.
* **Model**: one planar 3-bay braced frame (3D model, out-of-plane restrained) with its leaning column. That is the
  same per-frame basis as the design base shear `Vb` and weight `Wt`, so $\Omega = V_{max}/V_b$ is like-for-like.
* **Expected strength**: the model uses **expected** steel strength ($1.45 f_y$ for S235), which lifts $\Omega$ above
  the code's $q_S$.
* **Runtime**: a cyclic pushover to 6 % drift is far longer than a monotonic one. The total path is the sum of all
  48 half-cycle amplitudes, several metres of roof travel at 0.2 mm steps.

In [1]:
%load_ext autoreload
%autoreload 2

## 0. Setup & parameters

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from phd_project.config import config
from phd_project.scripts.cpo_du import resolve_cpo_du
from phd_project.scripts.loading_protocols import get_FEMA461_displacements_for_building
from phd_project.scripts.case_study_design_scripts.design_file_helpers import (
    get_control_node_from_design_file,
    get_n_damping_modes_from_design_file,
    get_n_primary_modes_from_design_file,
)
from phd_project.scripts.templates.copy_templates_to_folders import (
    configure_batch_run_file,
    copy_analysis_config,
    copy_file,
    copy_nlcbf_model,
)
from phd_project.scripts.WP1_ground_motion_set.design_groups import load_design_groups

cfg = config.load_config()

In [7]:
# ----------------------------------------------------------------------------
# PARAMETERS
# ----------------------------------------------------------------------------
# Root of the analysis folders - the same group tree as nb 052 / 054:
# DEST_ROOT/group_{n}s_{gid:02d}/{n}s/mdof/
DEST_ROOT = Path(cfg["analysis_data"]["wp1_fixed_record_sets"])

# The site-specific designs from nb 011; each group uses its representative's design.
DESIGN_ROOT = Path(cfg["models"]["casestudy_designs_site_specific"])

# The DEST_ROOT folders can only be written where the analysis drive is attached.
# False writes only the in-repo launchers (with the DEST_ROOT paths). SET THIS TO
# True ON THE ANALYSIS MACHINE.
BUILD_ANALYSIS_FOLDERS = True

STOREYS = [3, 5]

# --- structural model: must match nb 052 / 054 so the shared files stay identical ---
DAMPING_RATIO = 0.05
MDOF_DRIFT_LIMIT = 0.2

# --- cyclic pushover (FEMA 461), as nb 010 but with a drift-based amplitude ---
CPO_MAX_DRIFT = 6.0      # %, peak roof drift of the protocol (+/-); None -> the helper's own estimate
CPO_N_STEPS = 12         # FEMA 461 amplitude levels (a_i+1 = 1.4 a_i), as nb 010
CPO_DU = 0.2             # mm, base displacement step of the ramp, as nb 010
CPO_DU_OVERRIDES = {}    # {"group_3s_00/3s/mdof": dU} explicit per-folder choices
PRESERVE_TUNED_DU = True # keep a finer dU already tuned in a config on disk
CPO_RESULTS = "cyclic_pushover"   # results sub-folder
MODAL_RESULTS = "modal"

# --- batch launchers (heterogeneous-jobs template, as nb 010) ---
BATCH_BASE = Path(cfg["scripts"]["batch_run_analyses"]) / "fixed_record_sets"

print(f"DEST_ROOT  = {DEST_ROOT}  (build folders: {BUILD_ANALYSIS_FOLDERS})")
print(f"BATCH_BASE = {BATCH_BASE}")

DEST_ROOT  = D:\08_wp1_fixed_record_sets  (build folders: True)
BATCH_BASE = C:\Users\clemettn\Documents\phd\phd_project\scripts\WP1_ground_motion_set\batch_run_analyses\fixed_record_sets


## 1. The design groups

One analysis per design group (nb `051`): 25 × 3s and 26 × 5s. Each group is represented by its
`representative_tag`, whose design every member site shares.

In [8]:
groups_df = load_design_groups(cfg)
groups = (groups_df[groups_df["is_representative"] & groups_df["storeys"].isin(STOREYS)]
          .loc[:, ["storeys", "group_id", "representative_site", "representative_tag", "n_sites_in_group"]]
          .sort_values(["storeys", "group_id"])
          .reset_index(drop=True))
assert len(groups) == 51, f"{len(groups)} design groups, expected 51"


def group_folder_name(n: int, gid: int) -> str:
    # group_3s_00, ... - also the design_group_id used by nb 072
    return f"group_{n}s_{int(gid):02d}"


def mdof_folder(n: int, gid: int) -> Path:
    # DEST_ROOT/group_{n}s_{gid}/{n}s/mdof - identical to nb 052
    return DEST_ROOT / group_folder_name(n, gid) / f"{n}s" / "mdof"


print(groups.groupby("storeys").size().to_string())
groups.head()

storeys
3    25
5    26


,storeys,group_id,representative_site,representative_tag,n_sites_in_group
0,3,0,0,3s_cbf_dc2_site0,8
1,3,1,1,3s_cbf_dc2_site1,6
2,3,2,3,3s_cbf_dc2_site3,1
3,3,3,7,3s_cbf_dc2_site7,5
4,3,4,9,3s_cbf_dc2_site9,3


## 2. Write the modal and cyclic-pushover files into each group folder

The design file and the full-recorder model are (re)written exactly as nb `052` writes them, so these runs use the
same model as the group's IDA and MSA. Nothing else in the folder is touched.

In [9]:
def roof_height_mm(design_file: Path) -> float:
    # Roof height [mm]: the top entry of the design's level coordinates.
    with open(design_file) as f:
        return float(json.load(f)["structure"]["level_coordinates"][-1])


def cpo_displacements(design_file: Path) -> list[float]:
    # FEMA 461 target roof displacements [mm] for one structure, ending at 0.
    # U_max = CPO_MAX_DRIFT % of the roof height; None -> the helper's estimate.
    u_max = None if CPO_MAX_DRIFT is None else CPO_MAX_DRIFT / 100 * roof_height_mm(design_file)
    disp = get_FEMA461_displacements_for_building(design_file, CPO_N_STEPS, U_max=u_max)
    return np.round(disp, 3).tolist()


# The protocol of the first representative design, for inspection
_d0 = DESIGN_ROOT / groups.loc[0, "representative_tag"] / f"{groups.loc[0, 'representative_tag']}_out.json"
_p0 = cpo_displacements(_d0)
print(f"{groups.loc[0, 'representative_tag']}: roof height {roof_height_mm(_d0):.0f} mm, "
      f"{len(_p0)} targets, first +/-{_p0[0]:.1f} mm ... last +/-{max(abs(x) for x in _p0):.1f} mm, "
      f"total path {np.abs(np.diff([0] + _p0)).sum() / 1000:.1f} m")


def build_group_po_modal(n: int, gid: int, tag: str) -> Path:
    # Write model + modal + pushover into one group folder; return the folder.
    folder = mdof_folder(n, gid)
    folder.mkdir(parents=True, exist_ok=True)

    # --- design file (the model reads it from its own folder) ---
    design_src = DESIGN_ROOT / tag / f"{tag}_out.json"
    design_dst = folder / f"{tag}_designfile.json"
    copy_file(design_src, design_dst)

    # --- structural model, full recorders (same arguments as nb 052) ---
    init_fn_full = copy_nlcbf_model(
        cfg["templates"], folder,
        design_json=design_dst.name,
        damping_updates={"n_modes": get_n_damping_modes_from_design_file(design_dst),
                         "damping_ratio": DAMPING_RATIO},
        recorder_updates={"drift_limit": MDOF_DRIFT_LIMIT},
    )

    # --- modal analysis: periods and mode shapes of the primary modes ---
    copy_file(cfg["templates"]["run_modal"], folder / "run_modal.py")
    copy_analysis_config(
        cfg["templates"]["config_modal"], folder / "config_modal.py",
        results_folder_name=MODAL_RESULTS,
        model_file_name=init_fn_full,
        update_config={"n_modes": get_n_primary_modes_from_design_file(design_dst)},
    )

    # --- cyclic pushover: FEMA 461 protocol to +/- U_max at the roof, EC8
    # triangular pattern (template default) ---
    copy_file(cfg["templates"]["run_cyclic_pushover"], folder / "run_cyclic_pushover.py")
    cfg_dst = folder / "config_cyclic_pushover.py"
    # override > dU already tuned on disk > CPO_DU (see phd_project/scripts/cpo_du.py)
    du = resolve_cpo_du(folder.relative_to(DEST_ROOT).as_posix(), cfg_dst,
                        CPO_DU_OVERRIDES, CPO_DU, PRESERVE_TUNED_DU)
    copy_analysis_config(
        cfg["templates"]["config_cyclic_pushover"], cfg_dst,
        results_folder_name=CPO_RESULTS,
        model_file_name=init_fn_full,
        update_config={
            "displacement_type": "displacement",
            "dU": du,
            "ctrl_node": get_control_node_from_design_file(design_dst),
            "displacements": cpo_displacements(design_dst),
        },
    )
    return folder


folders = {}
for g in groups.itertuples(index=False):
    n, gid, tag = int(g.storeys), int(g.group_id), g.representative_tag
    if BUILD_ANALYSIS_FOLDERS:
        if not (DESIGN_ROOT / tag / f"{tag}_out.json").is_file():
            print(f"WARNING: skipping {group_folder_name(n, gid)} - design of {tag} not found (run nb 011)")
            continue
        folders[(n, gid)] = build_group_po_modal(n, gid, tag)
    else:
        folders[(n, gid)] = mdof_folder(n, gid)

verb = "built" if BUILD_ANALYSIS_FOLDERS else "listed (folders NOT built - BUILD_ANALYSIS_FOLDERS = False)"
print(f"{len(folders)} group folder(s) {verb}")

3s_cbf_dc2_site0: roof height 10500 mm, 49 targets, first +/-15.6 mm ... last +/-630.0 mm, total path 17.3 m
51 group folder(s) built


## 3. Batch launchers

`cpo.py` and `modal.py` per storey count, from the heterogeneous-jobs template (as nb `010`): each opens one console
per group, with concurrency capped by the process semaphore. The paths are always the `DEST_ROOT` ones.

In [10]:
def jobs(n: int, script: str, config_name: str, suffix: str) -> list[dict]:
    # One launcher entry per group of storey count n.
    return [{"script": folder / script,
             "config": [folder / config_name],
             "name": [f"{group_folder_name(nn, gid)}_{suffix}"]}
            for (nn, gid), folder in sorted(folders.items()) if nn == n]


for n in STOREYS:
    batch_dir = BATCH_BASE / f"{n}s" / "mdof"
    for filename, entries in [("cpo.py", jobs(n, "run_cyclic_pushover.py", "config_cyclic_pushover.py", "cpo")),
                              ("modal.py", jobs(n, "run_modal.py", "config_modal.py", "modal"))]:
        configure_batch_run_file(cfg["templates"]["batch_run"], batch_dir / filename, entries)
        print(f"wrote {(batch_dir / filename).relative_to(BATCH_BASE)} ({len(entries)} jobs)")

wrote 3s\mdof\cpo.py (25 jobs)
wrote 3s\mdof\modal.py (25 jobs)
wrote 5s\mdof\cpo.py (26 jobs)
wrote 5s\mdof\modal.py (26 jobs)


---

# ▶ RUN BARRIER - run the analyses (outside this notebook)

On the analysis machine (with `BUILD_ANALYSIS_FOLDERS = True` run first), from each
`batch_run_analyses/fixed_record_sets/{n}s/mdof/` folder:

```
python modal.py     # fast
python cpo.py       # FEMA 461 cyclic pushover to +/- 6 % roof drift - slow
```

Results land in each group folder:

* `modal/` - `modal_properties.json` (periods, participation) and `mode_shapes.csv`;
* `cyclic_pushover/po_curve.csv` - two columns: **control-node displacement [mm]** and **load factor = base shear
  [N]**, over the whole cyclic history; plus the recorders (`recorders.pickle`).

If a model fails to converge, set a finer `dU` for that folder in `CPO_DU_OVERRIDES` (key
`group_{n}s_{gid:02d}/{n}s/mdof`) and rerun section 2. Tuned values already on disk are preserved.

**From cyclic curve to properties (nb `056`).** Extract the **envelope** of the hysteresis: the peak base shear
reached at each new displacement excursion, positive and negative branches separately. The local `fitpo` package
has `fit_envelope` / `fit_piecewise_backbone`. Read $V_{max}$, $\delta_{y,eff}$ and $\delta_u$ from the envelope.
Use the positive branch, or the mean of the two branches, and record which. Strength lost *within* a cycle at
constant amplitude is cyclic degradation. It does not enter the envelope, but it is worth noting per design.

## What nb `056` must produce for nb `072`

`data_processed/08_casestudy_structure_datasets/design_group_structure_properties.csv`, one row per design group:

| column | meaning |
|---|---|
| `design_group_id` | `group_{n}s_{gid:02d}` |
| `T1_modal_s` | first-mode period from `modal/` [s] (nb 072 cross-checks it against the IDA's $T_1$) |
| `V_max_N` | peak base shear of the cyclic-pushover envelope [N] |
| `W_N` | seismic weight of the frame, `Wt` [N] |
| `Vb_design_N` | design base shear of the frame, `Vb` [N] |
| `delta_y_eff_mm` | effective yield roof displacement [mm] |
| `delta_u_mm` | roof displacement where the envelope has dropped to $0.8\,V_{max}$ after the peak [mm] |
| `roof_height_mm` | roof height [mm] (10 500 for 3s, 17 500 for 5s) |
| `overstrength` | $\Omega = V_{max}/V_{b}$ |
| `ductility` | $\mu_T = \delta_u/\delta_{y,eff}$ (FEMA P695 §6.3, section number approximate) |

Apply one definition of $\delta_{y,eff}$ to all 51 designs and record it in nb `056`: either the idealised elastic
line ($V_{max}/K_{el}$) or the FEMA P695 period-based form. If a cyclic pushover stops, or its envelope never
drops to $0.8\,V_{max}$ within $\pm U_{max}$, flag the design rather than extrapolate.